In [1]:

from notebooks.utils import find_project_root
import sqlite3
import sys
from pathlib import Path
from typing import Sequence

import pandas as pd


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DB_PATH = project_root / "data/nhs_provider_data.db"
DB_PATH_CONSOLIDATED = project_root / "data/nhs_consolidated.db"
DATA_DIR = "./data"

conn = sqlite3.connect(DB_PATH)
conn2 = sqlite3.connect(DB_PATH_CONSOLIDATED)

In [2]:


query = """
        SELECT i.period,
               i.provider_code                                                as provider,
               i.treatment_function_code                                      as treatment_code,
               i.treatment_function                                           as treatment,
               i.Total_number_of_incomplete_pathways                          as incomplete,
               LAG(i.Total_number_of_incomplete_pathways)                        OVER (
                PARTITION BY i.provider_code, i.treatment_function_code
                ORDER BY i.period
            ) AS inc_prev, dta.Total_number_of_incomplete_pathways_with_a_decision_to_admit_for_treatment as incomplete_dta,
               a.Total_number_of_completed_pathways_with_a_known_clock_start  as admitted,
               na.Total_number_of_completed_pathways_with_a_known_clock_start as nonadmitted,
               np.Number_of_new_RTT_clock_starts_during_the_month             as new_periods,
                    -- the accunting identity
                    (
                        i.Total_number_of_incomplete_pathways
                            - (
                            LAG(i.Total_number_of_incomplete_pathways)
                                OVER (PARTITION BY i.provider_code, i.treatment_function_code ORDER BY i.period)
                                + np.Number_of_new_RTT_clock_starts_during_the_month
                                - a.Total_number_of_completed_pathways_with_a_known_clock_start
                                - na.Total_number_of_completed_pathways_with_a_known_clock_start
                            )
                        )                                                                                AS residual,
               np.Number_of_new_RTT_clock_starts_during_the_month
                   - a.Total_number_of_completed_pathways_with_a_known_clock_start
                   - na.Total_number_of_completed_pathways_with_a_known_clock_start
                                                                              AS delta
        FROM incomplete AS i
                 INNER JOIN incomplete_with_dta AS dta ON i.period = dta.period
            AND i.provider_code = dta.provider_code
            AND i.treatment_function_code = dta.treatment_function_code
                 INNER JOIN new_periods np ON i.period = np.period
            AND i.provider_code = np.provider_code
            AND i.treatment_function_code = np.treatment_function_code
                 INNER JOIN admitted a ON i.period = a.period
            AND i.provider_code = a.provider_code
            AND i.treatment_function_code = a.treatment_function_code
                 INNER JOIN nonadmitted na ON i.period = na.period
            AND i.provider_code = na.provider_code
            AND i.treatment_function_code = na.treatment_function_code
        WHERE i.provider_code IN ('RAJ', 'RTH')
          AND treatment_code IN
              ('C_320', 'C_330', 'C_400', 'C_502')
        ORDER BY provider ASC, treatment_code ASC, i.period ASC; \
        """
#  'RJZ', 'RH5'
#               ('C_101', 'C_110', 'C_120', 'C_140', 'C_301',
#                'C_320', 'C_330', 'C_400', 'C_502')

df = pd.read_sql_query(query, conn,
                       # index_col="period"
                       )

df

,period,provider,treatment_code,treatment,incomplete,inc_prev,incomplete_dta,admitted,nonadmitted,new_periods,residual,delta
0,2019-01-01,RAJ,C_320,Cardiology,1602.0,NaN,30.0,17.0,846.0,788.0,NaN,-75.0
1,2019-02-01,RAJ,C_320,Cardiology,1615.0,1602.0,30.0,12.0,585.0,743.0,-133.0,146.0
2,2019-03-01,RAJ,C_320,Cardiology,1754.0,1615.0,23.0,24.0,652.0,844.0,-29.0,168.0
3,2019-04-01,RAJ,C_320,Cardiology,1825.0,1754.0,33.0,11.0,729.0,868.0,-57.0,128.0
4,2019-05-01,RAJ,C_320,Cardiology,2156.0,1825.0,33.0,31.0,563.0,870.0,55.0,276.0
...,...,...,...,...,...,...,...,...,...,...,...,...
635,2025-04-01,RTH,C_502,Gynaecology Service,6231.0,6000.0,1502.0,187.0,973.0,1436.0,-45.0,276.0
636,2025-05-01,RTH,C_502,Gynaecology Service,6280.0,6231.0,1565.0,232.0,986.0,1527.0,-260.0,309.0
637,2025-06-01,RTH,C_502,Gynaecology Service,6494.0,6280.0,1592.0,240.0,842.0,1425.0,-129.0,343.0
638,2025-07-01,RTH,C_502,Gynaecology Service,6847.0,6494.0,1565.0,217.0,891.0,1586.0,-125.0,478.0
